# Web Search Agent

A web search agent closes the knowledge gap between an LLM's training cutoff and today's world. By combining a fast search tool with a scraper that reads full page content, the agent can answer questions about current events, prices, documentation, and anything else that lives on the web.

**What you'll build:** an agent that uses DuckDuckGo for quick searches, scrapes full page content when it needs more detail, and optionally upgrades to Tavily for richer structured results.

**Time:** ~15 min | **Difficulty:** Beginner

**What you'll learn:**
- `DuckDuckGoSearchTool` — zero-API-key web search
- `WebScraperTool` — follow URLs and extract page text
- `TavilySearchTool` — structured search results with AI-optimized excerpts
- How to compose search + scraping for deeper research
- Switching between free and paid search providers

## Prerequisites

You need:
- An **OpenAI API key** (`OPENAI_API_KEY`)
- Optionally a **Tavily API key** (`TAVILY_API_KEY`) for richer search results
- The `synapsekit` package (installed in the next cell)

`DuckDuckGoSearchTool` works with no API key at all — great for getting started quickly.

In [ ]:
!pip install synapsekit -q

In [ ]:
import os

# Required
# os.environ['OPENAI_API_KEY'] = 'sk-...'

# Optional — for Tavily search
# os.environ['TAVILY_API_KEY'] = 'tvly-...'

## Step 1: Import tools

Import the search tools, scraper, agent class, and LLM.

In [ ]:
import asyncio
from synapsekit.agents import (
    FunctionCallingAgent,
    DuckDuckGoSearchTool,
    WebScraperTool,
    TavilySearchTool,
)
from synapsekit.llms.openai import OpenAILLM

## Step 2: Build the agent with free search

`DuckDuckGoSearchTool` requires no API key and is sufficient for most general queries. Pair it with `WebScraperTool` so the agent can follow links from search results and read the full page when a snippet is not enough.

In [ ]:
agent = FunctionCallingAgent(
    llm=OpenAILLM(model="gpt-4o-mini"),
    tools=[
        DuckDuckGoSearchTool(),
        WebScraperTool(),
    ],
    system_prompt=(
        "You are a research assistant. When answering questions about current events "
        "or facts that may have changed recently, always search the web first. "
        "If a search snippet is insufficient, scrape the full page for detail."
    ),
    max_iterations=6,
)

## Step 3: Run a current-events query

Ask the agent about something that changes over time — current Python version, latest news, etc.

In [ ]:
async def search(question: str) -> str:
    return await agent.run(question)

answer = asyncio.run(search("What is the current stable version of Python?"))
print(answer)

## Step 4: Upgrade to Tavily for richer results

Tavily returns AI-curated excerpts, source URLs, and relevance scores. Use it when you need cleaner, more structured search results — especially for technical topics where DuckDuckGo may surface low-quality pages.

In [ ]:
import os

# Use Tavily when the API key is available, fall back to DuckDuckGo otherwise
if os.getenv("TAVILY_API_KEY"):
    search_tool = TavilySearchTool()
else:
    search_tool = DuckDuckGoSearchTool()

premium_agent = FunctionCallingAgent(
    llm=OpenAILLM(model="gpt-4o-mini"),
    tools=[search_tool, WebScraperTool()],
    system_prompt="You are a research assistant with access to web search.",
    max_iterations=6,
)

print(f"Using search tool: {search_tool.__class__.__name__}")

## Step 5: Stream the agent's search process

Seeing which URLs the agent decides to visit and which it skips helps you understand whether your system prompt is steering it correctly.

In [ ]:
from synapsekit.agents import ActionEvent, FinalAnswerEvent, ObservationEvent

async def stream_search(question: str) -> None:
    async for event in agent.stream_steps(question):
        if isinstance(event, ActionEvent):
            tool_input = str(event.tool_input)
            print(f"[{event.tool}] {tool_input[:100]}")
        elif isinstance(event, ObservationEvent):
            print(f"  -> {event.observation[:150]}...")
        elif isinstance(event, FinalAnswerEvent):
            print(f"\nAnswer:\n{event.answer}")

asyncio.run(stream_search("What are the top AI announcements from the past month?"))

## Step 6: Batch multiple queries

For research tasks that require answering several related questions, run them sequentially and collect answers into a report.

In [ ]:
async def batch_research(questions: list[str]) -> dict[str, str]:
    results = {}
    for question in questions:
        results[question] = await agent.run(question)
    return results

results = asyncio.run(batch_research([
    "What is the current version of Python?",
    "What is the current version of Node.js?",
]))

for q, a in results.items():
    print(f"Q: {q}\nA: {a}\n")

## Complete Working Example

A self-contained script that builds an agent with automatic Tavily/DuckDuckGo selection, streams step events for two questions, and prints final answers.

In [ ]:
import asyncio
import os
from synapsekit.agents import (
    ActionEvent,
    DuckDuckGoSearchTool,
    FinalAnswerEvent,
    FunctionCallingAgent,
    ObservationEvent,
    TavilySearchTool,
    WebScraperTool,
)
from synapsekit.llms.openai import OpenAILLM


def build_agent() -> FunctionCallingAgent:
    # Prefer Tavily for richer results; DuckDuckGo requires no key
    search_tool = (
        TavilySearchTool() if os.getenv("TAVILY_API_KEY") else DuckDuckGoSearchTool()
    )
    return FunctionCallingAgent(
        llm=OpenAILLM(model="gpt-4o-mini"),
        tools=[search_tool, WebScraperTool()],
        system_prompt=(
            "You are a concise research assistant. Search the web for up-to-date information. "
            "Cite at least one source URL in every answer."
        ),
        max_iterations=6,
    )


async def main() -> None:
    agent = build_agent()

    questions = [
        "What is the current version of Python and when was it released?",
        "What are the top AI announcements from the past month?",
    ]

    for question in questions:
        print(f"\nQ: {question}")
        print("-" * 60)

        async for event in agent.stream_steps(question):
            if isinstance(event, ActionEvent):
                print(f"  Calling {event.tool}: {str(event.tool_input)[:80]}")
            elif isinstance(event, ObservationEvent):
                print(f"  Got: {event.observation[:120]}...")
            elif isinstance(event, FinalAnswerEvent):
                print(f"\n{event.answer}")


asyncio.run(main())

## Next Steps

- [ReAct Research Assistant](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/react-research-assistant) — add Wikipedia and arXiv alongside web search
- [Multi-Tool Orchestration](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/multi-tool-orchestration) — combine search with calculators, databases, and code execution
- [Agent with Safety Guardrails](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/agent-with-guardrails) — validate that output does not contain PII or blocked topics